In [2]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# database details
SERVER = r"Localhost\SQLEXPRESS"  # change to your server name
DATABASE = "VoltNestAnalytics"

# processed data folder
processed_dir = Path.cwd().parent / "data" / "processed"

# connect to sql server using windows authentication
connection_url = URL.create(
    "mssql+pyodbc",
    host=SERVER,
    database=DATABASE,
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "Trusted_Connection": "yes",
        "TrustServerCertificate": "yes"
    }
)

engine = create_engine(connection_url)

# import all csv files
for file in processed_dir.glob("*.csv"):
    df = pd.read_csv(file, low_memory=False)

    table_name = file.stem

    df.to_sql(
        name=table_name,
        con=engine,
        schema="dbo",
        if_exists="fail",
        index=False
    )

    print(f"{table_name}: {len(df):,} rows imported")

engine.dispose()

campaigns: 19 rows imported
customers: 3,302 rows imported
customer_summary: 3,302 rows imported
marketing_spend: 6,552 rows imported
orders: 4,278 rows imported
website_events: 144,097 rows imported
website_sessions: 45,000 rows imported


In [3]:
from sqlalchemy import create_engine, text

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'dbo'
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
    """))

    for row in result:
        print(row[0])

engine.dispose()

campaigns
customer_summary
customers
marketing_spend
orders
website_events
website_sessions
